# Jev vs Claude Opus: clinical coherence and calibration benchmark (v3)

Everything lives in `MyDrive/jev_bench/`. Code for this version is in `code/v3/` and is checked against fixed SHA-256 hashes before anything runs. See `CHANGELOG.md` for version history.

**Secrets** (key icon, left sidebar, notebook access on): `OPENROUTER_API_KEY`, `ANTHROPIC_API_KEY`.

Run top to bottom. Sections 6 to 8 skip themselves until your data files are in `jev_bench/data/`.

## 1. Setup

In [30]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys, shutil, hashlib
VERSION = 'v3'
WORK = '/content/drive/MyDrive/jev_bench'
CODE = f'{WORK}/code/{VERSION}'
D = f'{WORK}/data'
OUT = f'{WORK}/results.jsonl'
os.makedirs(D, exist_ok=True)

!pip -q install anthropic

for k in ('OPENROUTER_API_KEY', 'ANTHROPIC_API_KEY'):
    try:
        os.environ[k] = userdata.get(k)
        print(k, 'loaded')
    except Exception as e:
        print(k, 'MISSING:', e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OPENROUTER_API_KEY loaded
ANTHROPIC_API_KEY loaded


## 2. Load and verify the harness code
Copies `code/v3/` into the session and refuses to continue if any file differs from the version's recorded hash.

In [31]:
EXPECTED = {'clients.py': 'ef41cc0c4687c9c679522770c10903e0a83e8cb470e9177f4118480d3c59afec', 'run.py': '2d80d2dbbc106bff430478d1a3d10a576071740f4f85ce9b7c6873567c2a6f3d', 'analyze.py': 'c47273df89c53ac71ccf8859bc2ed6db482195e7f4d1fbdedb1a4a022301884e', 'ddxplus_to_cases.py': '0a2c9072ec480fe52e3eef4f420fcc2f37a71372d252c84a442e34a922912c51'}

def sha(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

RUN = '/content/jev_bench'
os.makedirs(RUN, exist_ok=True)
bad = []
for f, h in EXPECTED.items():
    p = f'{CODE}/{f}'
    if not os.path.exists(p):
        bad.append(f'{f}: missing')
    elif sha(p) != h:
        bad.append(f'{f}: hash mismatch ({sha(p)[:12]} != {h[:12]})')
    else:
        shutil.copy(p, f'{RUN}/{f}')
if bad:
    raise SystemExit('Code check FAILED, paste this to Claude:\n' + '\n'.join(bad))
os.chdir(RUN)
sys.path.insert(0, RUN)
print('All', len(EXPECTED), 'files verified for', VERSION)

All 4 files verified for v3


## 3. Version control

`MyDrive/jev_bench/` is a git repository. `snapshot(label)` records a manifest (code hashes, data-file hashes, results hash), saves a snapshot of this notebook as it ran, commits, and sets `HARNESS_COMMIT` so every result row carries it. Data files are hashed but never committed. A plain timestamped copy of each manifest also goes to `versions/`.

In [32]:
import json, subprocess, datetime, glob

def _git(*args):
    return subprocess.run(['git', '-C', WORK, *args], capture_output=True, text=True)

def _notebook_json():
    try:
        from google.colab import _message
        return _message.blocking_request('get_ipynb', request='', timeout_sec=30)['ipynb']
    except Exception as e:
        print('  (could not capture notebook contents:', e, ')')
        return None

def snapshot(label):
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    os.makedirs(f'{WORK}/runs', exist_ok=True)
    nbj = _notebook_json()
    if nbj:
        json.dump(nbj, open(f'{WORK}/runs/notebook_snapshot.ipynb', 'w'), indent=1)
    manifest = {'label': label, 'utc': stamp, 'version': VERSION,
                'code_sha256': {f: sha(f'{CODE}/{f}') for f in EXPECTED},
                'data_sha256': {os.path.relpath(p, WORK): sha(p)
                                for p in sorted(glob.glob(f'{D}/**/*', recursive=True)) if os.path.isfile(p)},
                'results_sha256': sha(OUT) if os.path.exists(OUT) else None}
    json.dump(manifest, open(f'{WORK}/runs/MANIFEST.json', 'w'), indent=2)
    vdir = f'{WORK}/versions/{stamp}_{label}'
    os.makedirs(vdir, exist_ok=True)
    shutil.copy(f'{WORK}/runs/MANIFEST.json', vdir)

    if not os.path.exists(f'{WORK}/.git'):
        _git('init', '-q')
        _git('config', 'user.name', 'jev_bench')
        _git('config', 'user.email', 'jev_bench@localhost')
        _git('config', 'core.fileMode', 'false')
        open(f'{WORK}/.gitignore', 'w').write('data/\nversions/\n')
    _git('add', '-A')
    c = _git('commit', '-q', '-m', f'{VERSION} {label} {stamp}')
    head = _git('rev-parse', '--short', 'HEAD').stdout.strip()
    if head:
        os.environ['HARNESS_COMMIT'] = head
        print(f'snapshot "{label}": commit {head}' + ('' if c.returncode == 0 else ' (no changes)'))
    else:
        h = hashlib.sha256(''.join(manifest['code_sha256'].values()).encode()).hexdigest()[:10]
        os.environ['HARNESS_COMMIT'] = f'nogit-{h}'
        print(f'snapshot "{label}": git unavailable, using code hash {h}')

snapshot('setup')
print(_git('log', '--oneline', '-5').stdout)

snapshot "setup": commit 022d0f0
022d0f0 v3 setup 20260927T021615Z
2b98c95 v3 setup 20260926T224533Z
e021379 v3 setup 20260926T224109Z



## 4. Offline smoke test (no API calls, no cost)

In [33]:
%%writefile example_options.json
{
  "toy_eye": {
    "options": {
      "retinal_detachment": "Retinal detachment",
      "posterior_vitreous_detachment": "Posterior vitreous detachment",
      "acute_angle_closure_glaucoma": "Acute angle-closure glaucoma",
      "central_retinal_artery_occlusion": "Central retinal artery occlusion",
      "bacterial_conjunctivitis": "Bacterial conjunctivitis",
      "anterior_uveitis": "Anterior uveitis",
      "corneal_abrasion": "Corneal abrasion",
      "dry_eye_disease": "Dry eye disease"
    },
    "distractors": {
      "ingrown_toenail": "Ingrown toenail",
      "benign_prostatic_hyperplasia": "Benign prostatic hyperplasia",
      "plantar_fasciitis": "Plantar fasciitis"
    }
  }
}


Overwriting example_options.json


In [34]:
%%writefile example_cases.csv
case_id,dataset,text,true_dx,paraphrase_1,paraphrase_2,paraphrase_3,distractors
t1,toy_eye,"Toy case: new flashes and a dark curtain coming over the vision in my right eye since yesterday.",retinal_detachment,"Toy case: since yesterday my right eye has flashing lights and a shade moving across.",,,ingrown_toenail;plantar_fasciitis
t2,toy_eye,"Toy case: sudden new floaters in my left eye, no change in vision otherwise, age 62.",posterior_vitreous_detachment,,,,ingrown_toenail
t3,toy_eye,"Toy case: severe eye pain, halos around lights, headache and vomiting tonight.",acute_angle_closure_glaucoma,,,,benign_prostatic_hyperplasia
t4,toy_eye,"Toy case: sudden painless total loss of vision in one eye an hour ago.",central_retinal_artery_occlusion,,,,plantar_fasciitis
t5,toy_eye,"Toy case: red eye with thick yellow discharge, lids stuck together this morning.",bacterial_conjunctivitis,,,,ingrown_toenail
t6,toy_eye,"Toy case: scratched my eye on a branch, now it waters and hurts to open.",corneal_abrasion,,,,ingrown_toenail


Overwriting example_cases.csv


In [35]:
%%writefile example_vague.csv
input_id,text
v1,"Toy input: my eye feels kind of off."
v2,"Toy input: something is not right, hard to explain."
v3,"Toy input: I just don't feel good today."


Overwriting example_vague.csv


In [36]:
!rm -f mock.jsonl
!python run.py --model mock --cases example_cases.csv --options example_options.json --vague example_vague.csv --out mock.jsonl
!python analyze.py mock.jsonl

88 jobs to run for mock (0 already done)
  50/88
# Results summary

Harness commit(s): 022d0f0
Resolved model versions: mock=mock-0

## Output resolution (Amendment 1, descriptive)

- mock: exact-zero rate 0.0%; values on a 0.01 grid 0.0%; smallest nonzero 0.0017 (n=712)

## mock / toy_eye
- H0: 60/60 repeat pairs byte-identical; TVD noise floor (p95) = 0.0000; log-ratio floor = 0.0000
- H1: violation rate 100.0% [61.0%, 100.0%] (n=6); top-1 flip rate 80.0%; mean max-TVD 0.626; Wilcoxon p=0.0156
- H2: violation rate 100.0% [20.7%, 100.0%] (n=1); top-1 flip rate 0.0%; mean max-TVD 0.365
- H3: violation rate 100.0% [61.0%, 100.0%] (n=6); mean distractor mass 0.0990; mean log-ratio shift 2.375
- H4: normalized entropy vague 0.680 vs informative 0.771; P(vague lower) test p=0.131; mean max-prob on vague 0.545
- H5: mean mass on none_of_these 0.081; mean max wrong-option mass 0.511; none_of_these is top in 0.0%
- H6: n 6; ECE 0.410; Brier 1.044; NLL_smoothed 2.859; NLL_raw 2.898; top1 0.167

## 5. Live checks (one Jev call, one Opus call; a few cents total)

In [37]:
import json
from collections import OrderedDict
from clients import JevClient, OpusClient
opts = OrderedDict(sorted(json.load(open('example_options.json'))['toy_eye']['options'].items()))
state = 'New flashes and a dark curtain coming over the vision in my right eye since yesterday.'
for name, client in (('Jev', JevClient()), ('Opus', OpusClient('verbalized'))):
    p, raw = client.classify(state, opts)
    print(f"{name} ({raw.get('resolved_model')}):")
    for k, v in sorted(p.items(), key=lambda x: -x[1])[:4]:
        print(f'  {v:.3f}  {k}')

Jev (typesafe/jev-1.13-20260917):
  0.910  retinal_detachment
  0.090  posterior_vitreous_detachment
  0.000  acute_angle_closure_glaucoma
  0.000  anterior_uveitis
Opus (claude-opus-5-5):
  0.874  retinal_detachment
  0.080  posterior_vitreous_detachment
  0.030  central_retinal_artery_occlusion
  0.005  acute_angle_closure_glaucoma


## 6. Study data

Files go in `MyDrive/jev_bench/data/`: `cases.csv`, `options.json`, `vague.csv` for the vignette sets, and `release_test_patients.csv`, `release_evidences.json`, `release_conditions.json` for DDXPlus.

In [38]:
def have(*paths):
    missing = [p for p in paths if not os.path.exists(p)]
    if missing:
        print('Skipping: not uploaded yet ->', ', '.join(os.path.basename(m) for m in missing))
    return not missing

sh = get_ipython().system

if have(f'{D}/release_test_patients.csv', f'{D}/release_evidences.json', f'{D}/release_conditions.json'):
    sh(f'python ddxplus_to_cases.py --patients {D}/release_test_patients.csv --evidences {D}/release_evidences.json '
       f'--conditions {D}/release_conditions.json --n 1000 --out {D}/ddxplus_cases.csv --options-out {D}/ddxplus_options.json')
    snapshot('ddxplus-converted')

Skipping: not uploaded yet -> release_test_patients.csv, release_evidences.json, release_conditions.json


## 7. Full runs
Appends to `results.jsonl` and resumes if interrupted. A snapshot is committed before each run. `--max-calls` refuses to start above the cap.

In [39]:
VIG = (f'{D}/cases.csv', f'{D}/options.json', f'{D}/vague.csv')
if have(*VIG):
    snapshot('pre-run-jev')
    sh(f'python run.py --model jev --cases {D}/cases.csv --options {D}/options.json --vague {D}/vague.csv --out {OUT} --max-calls 5000')

snapshot "pre-run-jev": commit 9ab7e87
570 jobs to run for jev (0 already done)
  50/570
  100/570
  150/570
  200/570
  250/570
  300/570
  350/570
  400/570
  450/570
  500/570
  550/570


In [40]:
if have(*VIG):
    snapshot('pre-run-opus-verbalized')
    sh(f'python run.py --model opus_verbalized --cases {D}/cases.csv --options {D}/options.json --vague {D}/vague.csv --out {OUT} --max-calls 5000')

snapshot "pre-run-opus-verbalized": commit ef4e15e
570 jobs to run for opus_verbalized (0 already done)
  50/570
  100/570
  150/570
  200/570
  250/570
  300/570
  350/570
  400/570
  450/570
  500/570
  550/570


In [41]:
if have(f'{D}/cases.csv', f'{D}/options.json'):
    snapshot('pre-run-opus-sampled')
    sh(f'python run.py --model opus_sampled --cases {D}/cases.csv --options {D}/options.json --out {OUT} --max-calls 5000')

snapshot "pre-run-opus-sampled": commit 6cb2181
45 jobs to run for opus_sampled (0 already done)


In [42]:
if have(f'{D}/ddxplus_cases.csv', f'{D}/ddxplus_options.json'):
    snapshot('pre-run-ddxplus')
    for m in ('jev', 'opus_verbalized'):
        sh(f'python run.py --model {m} --cases {D}/ddxplus_cases.csv --options {D}/ddxplus_options.json --out {OUT} --conditions H0 --max-calls 1100')

Skipping: not uploaded yet -> ddxplus_cases.csv, ddxplus_options.json


## 8. Analysis and final commit

In [43]:
if have(OUT):
    extra = f'--cases {D}/ddxplus_cases.csv' if os.path.exists(f'{D}/ddxplus_cases.csv') else ''
    sh(f'python analyze.py {OUT} {extra} > {WORK}/runs/summary.md')
    print(open(f'{WORK}/runs/summary.md').read())
    snapshot('analysis')
    print(_git('log', '--oneline', '-10').stdout)

# Results summary

Harness commit(s): 6cb2181, 9ab7e87, ef4e15e
Resolved model versions: jev=typesafe/jev-1.13-20260917, opus_sampled=claude-opus-5-5, opus_verbalized=claude-opus-5-5

## Output resolution (Amendment 1, descriptive)

- jev: exact-zero rate 94.0%; values on a 0.01 grid 99.6%; smallest nonzero 0.01 (n=21615)
- opus_sampled: exact-zero rate 97.4%; values on a 0.01 grid 100.0%; smallest nonzero 1 (n=1216)
- opus_verbalized: exact-zero rate 2.5%; values on a 0.01 grid 4.1%; smallest nonzero 0.0003 (n=21615)

## jev / semigran45
- H0: 295/450 repeat pairs byte-identical; TVD noise floor (p95) = 0.0600; log-ratio floor = 0.2895
- H1: violation rate 31.1% [19.5%, 45.7%] (n=45); top-1 flip rate 4.0%; mean max-TVD 0.061; Wilcoxon p=0.926
- H4: normalized entropy vague 0.317 vs informative 0.062; P(vague lower) test p=1; mean max-prob on vague 0.581
- H5: mean mass on none_of_these 0.566; mean max wrong-option mass 0.400; none_of_these is top in 53.3%
- H6: n 45; ECE 0.095; Brier 

In [44]:
import json, collections
rows = [json.loads(l) for l in open(OUT)]
print(collections.Counter(r['error'][:80] for r in rows if r.get('error')))
for r in rows:
    if r['model']=='jev' and r['condition']=='H4':
        top = max(r['probs'], key=r['probs'].get)
        print(r['case_id'], round(r['probs'][top],2), top)

Counter({"ValueError('all-zero distribution')": 13})
vg01 0.91 acute_conjunctivitis
vg02 0.51 stroke
vg03 0.86 acute_conjunctivitis
vg04 0.62 acute_conjunctivitis
vg05 0.93 acute_conjunctivitis
vg06 0.47 acute_conjunctivitis
vg07 0.74 acute_conjunctivitis
vg08 0.59 stroke
vg09 0.39 acute_sinusitis
vg10 0.89 acute_conjunctivitis
vg11 0.64 viral_upper_respiratory_illness
vg12 0.84 viral_upper_respiratory_illness
vg13 0.52 viral_upper_respiratory_illness
vg14 0.34 viral_upper_respiratory_illness
vg15 0.58 viral_upper_respiratory_illness
vg16 0.33 back_pain
vg17 0.43 viral_upper_respiratory_illness
vg18 0.4 viral_upper_respiratory_illness
vg19 0.22 back_pain
vg20 0.38 influenza
vg21 0.41 viral_upper_respiratory_illness
vg22 0.59 viral_upper_respiratory_illness
vg23 0.91 viral_upper_respiratory_illness
vg24 0.5 viral_upper_respiratory_illness
vg25 0.49 viral_upper_respiratory_illness
vg26 0.7 viral_upper_respiratory_illness
vg27 0.53 viral_upper_respiratory_illness
vg28 0.63 vertigo
vg29 0.